In [4]:
# INF201 - Deliverable 2
# Task 1:  File ingestion and regular expressions

from pathlib import Path
import re
import json

# Compiled once at module level
LOG_PATTERN = re.compile(
    r"\[(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\] "
    r"\[(?P<register_id>POS_\d+)\] "
    r"item=(?P<item>\w+) "
    r"price=(?P<price>\d+\.\d{2}) "
    r"(?P<currency>\w{3}) "
    r"\((?P<status>\w+)\)"
)


def filereader(file_path: str | Path) -> tuple[str, str]:
    """Reading a text file and it detects if it's UTF-8 or Latin-1
    
    UTF-8 is the first one to get tried, because that one is more
    strict and raises an error on invalid byte sequence. Latin-1
    on the other hand, accepts every byte, so it can never fail,
    so its used after
    
    Arguments: 
            file_path: Is a path to file, as a string or Path object
    Returns: 
            A tuple (file_text_content, encoding_used)
    """
    path = Path(file_path)
    try:
        file_text_content = path.read_text(encoding="utf-8")
        encoding_used = "utf-8"
    except UnicodeDecodeError:
        file_text_content = path.read_text(encoding="latin-1")
        encoding_used = "latin-1"
    return file_text_content, encoding_used


def log_parser(text: str) -> list[dict[str, str | float]]:
    """This function extracs the transaction lines from log text and 
    ignores system noise. Lines that dont match the sale lines like
    (printer, barcode, network messages) are skipped.
    
    Arguments: 
            text: The content of of the log file
    Returns: 
            A list of dictionaries with the keys, timestamp, register_id,
            item, price, currecny and status.
    """
    records = []
    for line in text.splitlines():
        transaction = LOG_PATTERN.match(line)
        if transaction is not None:
            dict_mode = transaction.groupdict()
            dict_mode["price"] = float(dict_mode["price"])
            records.append(dict_mode)
    return records


In [5]:
# INF201 - Deliverable 2
# Task 2:  Recursive utility functions

from typing import Any

def find_log_files(folder_path: str | Path) -> list[Path]:
    """Finding using recursive all the .log files in a folder and its subfolders
    
    The function uses path.iterdir() as asked, and avoids usage of .glob(), .rglob()
    and os.walk().
    
    Arguments:
            folder_path: the folder which the function starts searching in
            
    Returns:
            A list of different path objects, where each one has a .log found.
    """
    log_files = []
    path = Path(folder_path)
    for item in path.iterdir():
        if Path.is_dir(item):
            log_files.extend(find_log_files(item))
        elif Path.is_file(item) and item.suffix == ".log":
            log_files.append(item)
    return log_files


def load_config(config_path: str | Path) -> Any:
     """Load a JSON file and return back all the dicts ans lists.
     
     Arguments:
            config_path is a Path to the JSON file
    
     Returns:
            The JSON content
     """
     with open(config_path, "r", encoding="utf-8") as file:
        return json.load(file)


def active_register_ids(info: Any) -> set[str]:
    """The function uses recursive to collect all the ids of the active registers
    
    It goes through any combination of nested dicts and lists. Like a dictionary
    with "active": true and an "id" key as an active register. Everything else
    dosent get picked up, it gets skipped.
    
    Arguments:
            info: Is the part that gets inspected: dicts, lists or values
    
    Returns:
            returns a set of every id with an active register
    """
    active_ids: set[str] = set()
    if isinstance(info, dict):
        dict_active = info.get("active")
        if dict_active == True:
            try:
                 active_ids.add(info["id"])
            except KeyError:
                 print("The dictonary has no id with active:true")
        for thing in info.values():
             active_ids.update(active_register_ids(thing))
    elif isinstance(info,list):
         for thing in info:
              active_ids.update(active_register_ids(thing))
    return active_ids

In [6]:
# INF201 - Deliverable 2
# Task 3: Functions for cleaning and analysing data using NumPy

import numpy as np

# Remove price outliers
def remove_price_outliers(prices):
    """Replace outliers with the median."""
    prices = np.array(prices, dtype=float).flatten()

    outliers = abs(prices - prices.mean()) > 3 * prices.std()
    prices[outliers] = np.median(prices[~outliers])

    return prices


# Calculate refund ratio
def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    """Calculate refund ratio and avoid division by zero."""
    return np.asarray(refund_count) / (np.asarray(sales_count) + 1e-6)

In [ ]:
# INF201 - Deliverable 2
# Task 4: Data analysis using Pandas

import pandas as pd

# Create DataFrame
def build_dataframe(transactions):
    """Create a DataFrame sorted by timestamp."""
    df = pd.DataFrame(transactions)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    return df.set_index("timestamp").sort_index()


# Calculate hourly revenue
def hourly_revenue(df):
    """Sum prices per item every hour."""
    return df.groupby("item").resample("1h")["price"].sum().reset_index()


# Create pivot table
def status_pivot(df):
    """Sum prices by item and status."""
    return df.pivot_table(
        index="item",
        columns="status",
        values="price",
        aggfunc="sum",
        fill_value=0
    ).reindex(columns=["SUCCESS", "REFUND", "CANCELLED"], fill_value=0)


# Calculate refund ratio per item
def item_refund_ratio(df):
    """Calculate refund ratio for each item."""

    # Count transactions by item and status
    counts = df.groupby(["item", "status"]).size().unstack(fill_value=0)

    counts = counts.reindex(columns=["SUCCESS", "REFUND"], fill_value=0)

    # Calculate refund ratio
    counts["refund_ratio"] = compute_refund_ratio(counts["SUCCESS"].to_numpy(), counts["REFUND"].to_numpy())

    return counts[["refund_ratio"]]


Task 4 loaded successfully!


In [11]:
# INF201 - Deliverable 2
# Task 5:  Create full workflow

from pathlib import Path

def run_pipeline(data_folder="data"):
    """Read, clean, analyze and save transaction data."""

    # Find log files and active registers
    data_folder = Path(data_folder)
    log_files = find_log_files(data_folder)

    config = load_config(data_folder / "store_config.json")
    active_ids = active_register_ids(config)

    # Read and parse transactions
    transactions = []

    for file in log_files:
        text, encoding = filereader(file)
        records = log_parser(text)

        for record in records:
            if record["register_id"] in active_ids: transactions.append(record)

    if not transactions:
        raise ValueError("No transactions found.")

    # Create DataFrame
    df = build_dataframe(transactions)

    # Remove price outliers
    df["price"] = remove_price_outliers(df["price"])

    # Correct NOK prices
    df.loc[df["currency"] == "NOK", "price"] *= 10.0

    # Round prices to 2 decimal places
    df["price"] = df["price"].round(2)

    # Generate reports
    hourly = hourly_revenue(df)
    pivot = status_pivot(df)
    refunds = item_refund_ratio(df)

    # Save output files
    df.to_csv("cleaned_transactions.csv", float_format="%.2f")

    hourly.to_json("hourly_revenue.json", orient="records", date_format="iso", indent=2)

    pivot.to_csv("status_pivot_table.csv", float_format="%.2f")
    refunds.to_csv("item_refund_ratios.csv", float_format="%.2f")

    print("All files saved successfully!")

    return df


# Run the workflow
df = run_pipeline()

All files saved successfully!
